# TP — Classification de Texte
### Dataset : Requêtes clients bancaires (Banking77)

---

## Contexte

Une banque reçoit des milliers de messages clients chaque jour :
- *"My card was declined at the ATM"*
- *"I want to transfer money to my friend"*
- *"Someone made a payment I don't recognize"*

**Objectif** : entraîner un modèle qui lit le message et le redirige automatiquement vers le bon service.

Nous allons classer chaque message dans l'une de ces 3 catégories :

| Classe | Description | Exemples |
|---|---|---|
| **Paiement** | Opérations de transfert, solde, recharge | virement, top-up, bénéficiaire |
| **Compte** | Gestion de la carte et du compte | carte expirée, PIN, mot de passe |
| **Litige** | Problèmes, fraudes, remboursements | carte bloquée, fraude, remboursement |

> **Prérequis** : TP Text Mining (prétraitement, vectorisation TF-IDF)

---
# PARTIE 1 — Théorie

---

## 1.1 — Qu'est-ce que la classification de texte ?

La **classification de texte** est une tâche supervisée : on dispose d'exemples étiquetés (texte → classe) et on entraîne un modèle à apprendre cette correspondance pour prédire la classe de nouveaux textes.

```
ENTRAÎNEMENT :
"My card was declined"        → Litige
"I want to make a transfer"   → Paiement
"I forgot my PIN"             → Compte
         ↓
      Modèle
         ↓
PRÉDICTION :
"Someone used my card"        → ? (Litige)
```

Dans ce TP, nous allons tester et comparer **3 algorithmes** : Naïve Bayes, SVM, et Random Forest.

## 1.2 — Naïve Bayes (NB)

### Principe

Naïve Bayes est un classifieur probabiliste basé sur le **théorème de Bayes** :

$$P(classe\ |\ texte) = \frac{P(texte\ |\ classe) \times P(classe)}{P(texte)}$$

En pratique, pour chaque classe $c$ et chaque texte $d$ composé de mots $w_1, w_2, ..., w_n$ :

$$P(c\ |\ d) \propto P(c) \times \prod_{i=1}^{n} P(w_i\ |\ c)$$

L'hypothèse **naïve** : les mots sont **indépendants** entre eux. Faux en réalité, mais ça marche bien en pratique !

### Exemple

```
Texte : "card declined payment"

P(Litige  | texte) ∝ P(Litige)  × P(card|Litige)  × P(declined|Litige)  × P(payment|Litige)
P(Paiement| texte) ∝ P(Paiement)× P(card|Paiement)× P(declined|Paiement)× P(payment|Paiement)
P(Compte  | texte) ∝ P(Compte)  × P(card|Compte)  × P(declined|Compte)  × P(payment|Compte)

→ On choisit la classe avec la plus haute probabilité
```

### Hyperparamètre clé

| Paramètre | Rôle | Valeurs typiques |
|---|---|---|
| `alpha` | Lissage de Laplace — évite P=0 pour les mots non vus | 0.1, 0.5, 1.0 (défaut) |

> Un `alpha` faible → le modèle fait plus confiance aux données. Un `alpha` élevé → plus de régularisation.

## 1.3 — SVM (Support Vector Machine)

### Principe

Le SVM cherche l'**hyperplan** qui sépare les classes avec la **marge maximale**. En classification de texte, chaque document est un vecteur dans un espace de haute dimension (une dimension par mot du vocabulaire).

```
Espace vectoriel (simplifié à 2D) :

         ▲ mot "fraud"
   L  L  │  L
   L    ─┼─────────── hyperplan optimal
          │  P  P
        P │       P
          └──────────► mot "transfer"

L = Litige, P = Paiement
La marge entre l'hyperplan et les points les plus proches est maximisée.
```

### Astuce : le Kernel Trick

Quand les classes ne sont pas linéairement séparables, le SVM utilise une **fonction noyau (kernel)** pour projeter les données dans un espace de dimension supérieure où elles deviennent séparables.

### Hyperparamètres clés

| Paramètre | Rôle | Valeurs typiques |
|---|---|---|
| `C` | Pénalité pour les erreurs de classification. C élevé = moins d'erreurs tolérées = risque de surapprentissage | 0.1, 1, 10, 100 |
| `kernel` | Fonction de transformation de l'espace | `linear`, `rbf`, `poly` |
| `gamma` | Portée d'influence de chaque exemple (kernel RBF) | `scale`, `auto`, 0.1, 0.01 |

> Pour le texte, `kernel='linear'` est souvent suffisant et beaucoup plus rapide.

## 1.4 — Random Forest (RF)

### Principe

Random Forest construit un **ensemble d'arbres de décision** entraînés sur des sous-échantillons aléatoires des données et des features. La prédiction finale est un **vote majoritaire** de tous les arbres.

```
                 Texte
                /  |  \
           Arbre1 Arbre2 Arbre3 ... Arbre_N
             ↓      ↓      ↓
           Litige  Litige Compte
                     ↓
             Vote majoritaire
                     ↓
                   Litige  ✓
```

### Hyperparamètres clés

| Paramètre | Rôle | Valeurs typiques |
|---|---|---|
| `n_estimators` | Nombre d'arbres. Plus = meilleur mais plus lent | 100, 200, 500 |
| `max_depth` | Profondeur maximale de chaque arbre. Limiter évite le surapprentissage | None, 10, 20 |
| `max_features` | Nombre de features considérées à chaque split | `sqrt`, `log2`, 0.5 |

> Random Forest est robuste et nécessite peu de prétraitement, mais plus lent que NB et SVM sur du texte.

## 1.5 — Pipeline classique de classification de texte

```
Texte brut
    ↓
Prétraitement (nettoyage, tokenisation, stopwords)
    ↓
Vectorisation TF-IDF
(texte → vecteur numérique)
    ↓
Séparation Train / Test
    ↓
Entraînement du modèle
(Naïve Bayes / SVM / Random Forest)
    ↓
Évaluation
(Accuracy, Precision, Recall, F1)
    ↓
Comparaison des modèles
```

---
# PARTIE 2 — Mise en pratique

---

## Étape 0 — Installation

In [ ]:
!pip install datasets --quiet

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print('Tout est pret.')

---
## Étape 1 — Chargement du dataset

**Banking77** contient 13 083 messages clients bancaires répartis en 77 intentions fines.
Nous allons les regrouper en **3 grandes classes** pour notre tâche.

In [ ]:
from datasets import load_dataset

ds = load_dataset('PolyAI/banking77')
print(ds)

In [ ]:
# Conversion en DataFrame
df_train = pd.DataFrame(ds['train'])
df_test  = pd.DataFrame(ds['test'])
df = pd.concat([df_train, df_test], ignore_index=True)

# Recuperer les noms des intentions
label_names = ds['train'].features['label'].names
df['intent'] = df['label'].apply(lambda x: label_names[x])

print(f'Nombre de messages : {len(df)}')
print(f'Nombre d intentions : {df["intent"].nunique()}')
df.head(5)

In [ ]:
# Afficher toutes les intentions disponibles
print('Liste des 77 intentions :')
for i, name in enumerate(label_names):
    print(f'  {i:2d}. {name}')

---
## Étape 2 — Regroupement en 3 classes

On regroupe les 77 intentions en 3 grandes familles métier.
C'est ce qu'on ferait en entreprise pour router les messages vers les bons services.

In [ ]:
# Definition des 3 classes
PAIEMENT = [
    'transfer_internationally', 'transfer_not_received_by_recipient',
    'transfer_timing', 'transfer_fee_charged', 'transfer_into_account',
    'top_up_by_bank_transfer_charge', 'top_up_by_card_charge',
    'top_up_by_cash_or_cheque', 'top_up_failed', 'top_up_limits',
    'top_up_reverted', 'balance_not_updated_after_bank_transfer',
    'balance_not_updated_after_cheque_or_cash_deposit',
    'beneficiary_not_allowed', 'exchange_rate',
    'exchange_via_app', 'exchange_charge', 'fiat_currency_support',
    'pending_transfer', 'pending_top_up', 'receiving_money',
    'request_refund', 'supported_cards_and_currencies'
]

COMPTE = [
    'card_arrival', 'card_linking', 'card_about_to_expire',
    'card_acceptance', 'card_delivery_estimate',
    'pin_blocked', 'change_pin', 'activate_my_card',
    'age_limit', 'apple_pay_or_google_pay',
    'atm_support', 'automatic_top_up', 'contactless_not_working',
    'country_support', 'direct_debit_payment_not_recognised',
    'edit_personal_details', 'get_disposable_virtual_card',
    'get_physical_card', 'lost_or_stolen_card',
    'lost_or_stolen_phone', 'order_physical_card',
    'passcode_forgotten', 'verify_my_identity', 'verify_top_up',
    'virtual_card_not_working', 'visa_or_mastercard',
    'reverted_card_payment?', 'terminate_account',
    'spending_controls'
]

LITIGE = [
    'card_not_working', 'card_payment_fee_charged',
    'card_payment_not_recognised', 'card_payment_wrong_exchange_rate',
    'card_swallowed', 'cash_withdrawal_charge',
    'cash_withdrawal_not_recognised', 'compromised_card',
    'declined_card_payment', 'declined_cash_withdrawal',
    'declined_transfer', 'extra_charge_on_statement',
    'failed_transfer', 'fraud_dispute',
    'get_refund', 'pending_card_payment',
    'pending_cash_withdrawal', 'refund_not_showing_in_my_account',
    'wrong_amount_of_cash_received', 'wrong_exchange_rate_for_cash_withdrawal',
    'transaction_charged_twice', 'unable_to_verify_identity',
    'disposable_card_limits', 'banking_crisis'
]

print('Classes definies.')

In [ ]:
# Fonction de mapping intention -> classe
def mapper_classe(intent):
    if intent in PAIEMENT:
        return 'Paiement'
    elif intent in COMPTE:
        return 'Compte'
    elif intent in LITIGE:
        return 'Litige'
    else:
        return None  # intentions non assignees

df['classe'] = df['intent'].apply(mapper_classe)

In [ ]:
# Supprimer les messages non assignes
df = df[df['classe'].notna()].reset_index(drop=True)

print(f'Messages conserves : {len(df)}')
print()
print('Distribution des classes :')
print(df['classe'].value_counts())

In [ ]:
# Visualisation de la distribution
counts = df['classe'].value_counts()

plt.figure(figsize=(7, 4))
counts.plot(kind='bar', color=['#3498DB', '#E74C3C', '#2ECC71'])
plt.title('Distribution des 3 classes')
plt.ylabel('Nombre de messages')
plt.xticks(rotation=0)
for i, v in enumerate(counts.values):
    plt.text(i, v + 20, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Exemples par classe
print('=== Exemples de messages par classe ===\n')
for classe in ['Paiement', 'Compte', 'Litige']:
    exemples = df[df['classe'] == classe]['text'].sample(3, random_state=42).tolist()
    print(f'[{classe}]')
    for ex in exemples:
        print(f'  - {ex}')
    print()

---
## Étape 3 — Prétraitement

Les messages sont courts et déjà assez propres. On applique un nettoyage léger.

In [ ]:
stop_en = set(stopwords.words('english'))

In [ ]:
def nettoyer(texte):
    texte = re.sub(r'[^a-zA-Z\s]', ' ', texte)  # ponctuation et chiffres
    texte = re.sub(r'\s+', ' ', texte)           # espaces multiples
    return texte.strip().lower()

In [ ]:
df['text_clean'] = df['text'].apply(nettoyer)

# Verification
print('AVANT :', df['text'].iloc[0])
print('APRES :', df['text_clean'].iloc[0])

---
## Étape 4 — Vectorisation TF-IDF

On transforme chaque message en vecteur numérique avec TF-IDF.

- `max_features` : on garde les 5000 mots les plus informatifs
- `ngram_range=(1,2)` : on utilise les mots seuls ET les bigrammes (ex: "card declined")

In [ ]:
X = df['text_clean']
y = df['classe']

In [ ]:
# Separation train / test
# stratify=y garantit la meme proportion de classes dans train et test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Taille train : {len(X_train)}')
print(f'Taille test  : {len(X_test)}')

In [ ]:
# Vectorisation TF-IDF
# IMPORTANT : fit sur train seulement, transform sur train ET test
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words=list(stop_en)
)

X_train_vec = tfidf.fit_transform(X_train)
X_test_vec  = tfidf.transform(X_test)

print(f'Dimension des vecteurs : {X_train_vec.shape}')
print(f'({X_train_vec.shape[0]} messages, {X_train_vec.shape[1]} features TF-IDF)')

---
## Étape 5 — Modèle 1 : Naïve Bayes

On commence par Naïve Bayes, le plus simple et le plus rapide.

In [ ]:
# Entrainement
nb = MultinomialNB(alpha=1.0)  # alpha=1.0 : lissage de Laplace standard
nb.fit(X_train_vec, y_train)

In [ ]:
# Prediction
y_pred_nb = nb.predict(X_test_vec)

In [ ]:
# Evaluation
acc_nb = accuracy_score(y_test, y_pred_nb)
print(f'Accuracy Naive Bayes : {acc_nb*100:.2f}%')
print()
print(classification_report(y_test, y_pred_nb))

In [ ]:
# Matrice de confusion
cm_nb = confusion_matrix(y_test, y_pred_nb, labels=['Paiement', 'Compte', 'Litige'])

plt.figure(figsize=(6, 4))
sns.heatmap(cm_nb, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Paiement', 'Compte', 'Litige'],
            yticklabels=['Paiement', 'Compte', 'Litige'])
plt.title('Matrice de confusion — Naive Bayes')
plt.ylabel('Reel')
plt.xlabel('Predit')
plt.tight_layout()
plt.show()

> **Comment lire la matrice de confusion ?**
> - La **diagonale** = prédictions correctes
> - Les cases hors diagonale = erreurs
> - Exemple : ligne Litige / colonne Paiement = messages Litige classés à tort en Paiement

---
## Étape 6 — Modèle 2 : SVM

On utilise `LinearSVC` — version linéaire du SVM, très efficace sur le texte.

In [ ]:
# Entrainement
# C=1.0 : penalite standard (bon point de depart)
svm = LinearSVC(C=1.0, max_iter=2000, random_state=42)
svm.fit(X_train_vec, y_train)

In [ ]:
# Prediction et evaluation
y_pred_svm = svm.predict(X_test_vec)
acc_svm = accuracy_score(y_test, y_pred_svm)
print(f'Accuracy SVM : {acc_svm*100:.2f}%')
print()
print(classification_report(y_test, y_pred_svm))

In [ ]:
# Matrice de confusion
cm_svm = confusion_matrix(y_test, y_pred_svm, labels=['Paiement', 'Compte', 'Litige'])

plt.figure(figsize=(6, 4))
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Paiement', 'Compte', 'Litige'],
            yticklabels=['Paiement', 'Compte', 'Litige'])
plt.title('Matrice de confusion — SVM')
plt.ylabel('Reel')
plt.xlabel('Predit')
plt.tight_layout()
plt.show()

---
## Étape 7 — Modèle 3 : Random Forest

In [ ]:
# Entrainement
# n_estimators=200 : 200 arbres (bon compromis vitesse/performance)
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train_vec, y_train)

In [ ]:
# Prediction et evaluation
y_pred_rf = rf.predict(X_test_vec)
acc_rf = accuracy_score(y_test, y_pred_rf)
print(f'Accuracy Random Forest : {acc_rf*100:.2f}%')
print()
print(classification_report(y_test, y_pred_rf))

In [ ]:
# Matrice de confusion
cm_rf = confusion_matrix(y_test, y_pred_rf, labels=['Paiement', 'Compte', 'Litige'])

plt.figure(figsize=(6, 4))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Paiement', 'Compte', 'Litige'],
            yticklabels=['Paiement', 'Compte', 'Litige'])
plt.title('Matrice de confusion — Random Forest')
plt.ylabel('Reel')
plt.xlabel('Predit')
plt.tight_layout()
plt.show()

---
## Étape 8 — Comparaison des 3 modèles

In [ ]:
# Tableau recapitulatif
from sklearn.metrics import f1_score

resultats = pd.DataFrame({
    'Modele'   : ['Naive Bayes', 'SVM', 'Random Forest'],
    'Accuracy' : [acc_nb, acc_svm, acc_rf],
    'F1 macro' : [
        f1_score(y_test, y_pred_nb,  average='macro'),
        f1_score(y_test, y_pred_svm, average='macro'),
        f1_score(y_test, y_pred_rf,  average='macro')
    ]
})

resultats['Accuracy'] = (resultats['Accuracy'] * 100).round(2)
resultats['F1 macro'] = (resultats['F1 macro'] * 100).round(2)
print(resultats.to_string(index=False))

In [ ]:
# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

modeles = ['Naive Bayes', 'SVM', 'Random Forest']
couleurs = ['#3498DB', '#E74C3C', '#2ECC71']

# Accuracy
axes[0].bar(modeles, resultats['Accuracy'], color=couleurs)
axes[0].set_title('Accuracy par modele (%)')
axes[0].set_ylim([0, 100])
for i, v in enumerate(resultats['Accuracy']):
    axes[0].text(i, v + 0.5, f'{v}%', ha='center', fontweight='bold')

# F1 Score
axes[1].bar(modeles, resultats['F1 macro'], color=couleurs)
axes[1].set_title('F1 Score macro par modele (%)')
axes[1].set_ylim([0, 100])
for i, v in enumerate(resultats['F1 macro']):
    axes[1].text(i, v + 0.5, f'{v}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Matrices de confusion cote a cote
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
classes = ['Paiement', 'Compte', 'Litige']

for ax, cm, titre, cmap in zip(
    axes,
    [cm_nb, cm_svm, cm_rf],
    ['Naive Bayes', 'SVM', 'Random Forest'],
    ['Blues', 'Oranges', 'Greens']
):
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax,
                xticklabels=classes, yticklabels=classes)
    ax.set_title(f'Matrice de confusion\n{titre}')
    ax.set_ylabel('Reel')
    ax.set_xlabel('Predit')

plt.tight_layout()
plt.show()

---
## Étape 9 — Tester sur de nouveaux messages

On teste le meilleur modèle sur des messages inventés pour vérifier son comportement.

In [ ]:
# Choisir le meilleur modele
meilleur_modele = max(
    [('Naive Bayes', nb, acc_nb),
     ('SVM', svm, acc_svm),
     ('Random Forest', rf, acc_rf)],
    key=lambda x: x[2]
)
print(f'Meilleur modele : {meilleur_modele[0]} ({meilleur_modele[2]*100:.2f}%)')
modele_final = meilleur_modele[1]

In [ ]:
# Nouveaux messages a tester
nouveaux_messages = [
    "I tried to pay at the supermarket but my card was declined",
    "How do I change my PIN number?",
    "I want to send money to my friend in France",
    "Someone made a payment I don't recognize on my account",
    "My card hasn't arrived yet, it's been 10 days",
    "I need a refund for a wrong transaction"
]

In [ ]:
# Pretraitement et vectorisation
messages_clean = [nettoyer(m) for m in nouveaux_messages]
messages_vec   = tfidf.transform(messages_clean)

# Prediction
predictions = modele_final.predict(messages_vec)

print('=== Predictions sur nouveaux messages ===\n')
for msg, pred in zip(nouveaux_messages, predictions):
    print(f'Message : {msg}')
    print(f'Classe  : {pred}')
    print()

---
## Bilan

| Modèle | Points forts | Points faibles |
|---|---|---|
| **Naïve Bayes** | Très rapide, peu de données suffisent | Hypothèse d'indépendance irréaliste |
| **SVM** | Excellent sur le texte, robuste | Moins interprétable, lent sur grands corpus |
| **Random Forest** | Robuste, pas de normalisation nécessaire | Lent sur données de haute dimension (texte) |

**Points clés à retenir :**
- Le **SVM linéaire** est généralement le meilleur choix pour la classification de texte.
- **Naïve Bayes** est un excellent baseline : rapide et souvent surprenant.
- **Random Forest** est moins adapté au texte (haute dimension) mais reste compétitif.
- La **matrice de confusion** révèle quelles classes se confondent — plus utile que l'accuracy seule.
- Le paramètre `C` du SVM et `alpha` de Naïve Bayes sont les plus importants à régler.